In [1]:
# Encerra qualquer sessão antiga na memória
try:
    spark.stop()
except:
    pass

from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

builder = SparkSession.builder \
    .appName("FinancialDataLake") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.sql.shuffle.partitions", "4") \
    .master("local[*]")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

print("⚡ Spark", spark.version, "com Delta Lake ATIVADO com sucesso!")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/11 20:08:31 WARN Utils: Your hostname, Erior, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/11 20:08:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/home/ericl/projetos/datalake-pyspark/.venv/lib/python3.14/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /home/ericl/.ivy2.5.2/cache
The jars for the packages stored in: /home/ericl/.ivy2.5.2/jars
io.delta#delta-spark_4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8b7b91c3-ecfd-40da-8cc8-3415aad4d328;1.0
	confs: [default]
	found io.delta#delta-spark_4.1_2.13;4.3.1 in central
	found io.delta#delta-storage;4.3.1 in central
	found io.unitycatalog#unitycatalog-client;0.5.0 in central
	found org.slf4j#slf4j-api;2.0.13 in central
	fou

⚡ Spark 4.1.1 com Delta Lake ATIVADO com sucesso!


In [2]:
import yfinance as yf
import pandas as pd
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, TimestampType

# 1. Schema Enforcement (Contrato da Camada Bronze)
BRONZE_SCHEMA = StructType([
    StructField("Date",                StringType(),    True),
    StructField("Open",                DoubleType(),    True),
    StructField("High",                DoubleType(),    True),
    StructField("Low",                 DoubleType(),    True),
    StructField("Close",               DoubleType(),    True),
    StructField("Volume",              LongType(),      True),
    StructField("ticker_code",         StringType(),    False),
    StructField("ingestion_timestamp", TimestampType(), True),
])

# 2. Os 12 Maiores Ativos da B3 e do Mercado Global (Nasdaq / S&P 500)
TICKERS = [
    "PETR4.SA", "VALE3.SA", "ITUB4.SA", "BBDC4.SA", "WEGE3.SA", "ABEV3.SA",
    "AAPL", "NVDA", "MSFT", "AMZN", "GOOGL", "TSLA"
]
PATH_BRONZE = "./storage/bronze"

# 3. Extrair via yfinance selecionando APENAS as colunas do Schema
records = []
for ticker in TICKERS:
    print(f"📦 Extraindo API: {ticker}...")
    df = yf.Ticker(ticker).history(period="1y").reset_index()
    if df.empty:
        print(f"⚠️ Sem dados para: {ticker}")
        continue
    df["ticker_code"] = ticker
    df["ingestion_timestamp"] = datetime.now()
    df["Date"] = df["Date"].astype(str)
    
    # Filtra rigorosamente as 8 colunas exatas do Schema
    df_filtered = df[["Date", "Open", "High", "Low", "Close", "Volume", "ticker_code", "ingestion_timestamp"]]
    records.append(df_filtered)

df_all = pd.concat(records, ignore_index=True)

# 4. Criar DataFrame PySpark em Memória RAM
df_spark = spark.createDataFrame(df_all, schema=BRONZE_SCHEMA)

# 5. Salvar no HD como Delta Table (Persistente + ACID + Particionado por Ticker)
df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ticker_code") \
    .save(PATH_BRONZE)

print(f"\n🎉 CAMADA BRONZE GRAVADA COM SUCESSO!")
print(f"📊 Total de registros gravados no Delta Lake: {df_spark.count()}")

# 6. Inspecionar o Schema e as primeiras linhas no PySpark
df_spark.printSchema()
df_spark.show(5)


📦 Extraindo API: PETR4.SA...
📦 Extraindo API: VALE3.SA...
📦 Extraindo API: ITUB4.SA...
📦 Extraindo API: BBDC4.SA...
📦 Extraindo API: WEGE3.SA...
📦 Extraindo API: ABEV3.SA...
📦 Extraindo API: AAPL...
📦 Extraindo API: NVDA...
📦 Extraindo API: MSFT...
📦 Extraindo API: AMZN...
📦 Extraindo API: GOOGL...
📦 Extraindo API: TSLA...


26/08/11 20:08:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/08/11 20:08:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/08/11 20:08:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/08/11 20:08:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 69.09% for 11 writers
26/08/11 20:08:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 63.33% for 12 writers
26/08/11 20:08:46 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 58.46% for 13 writers
26/08/11 20:08:46 WARN MemoryManager: Total allocation exceeds 95.


🎉 CAMADA BRONZE GRAVADA COM SUCESSO!
📊 Total de registros gravados no Delta Lake: 3013
root
 |-- Date: string (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: long (nullable = true)
 |-- ticker_code: string (nullable = false)
 |-- ingestion_timestamp: timestamp (nullable = true)

+--------------------+------------------+------------------+------------------+------------------+--------+-----------+--------------------+
|                Date|              Open|              High|               Low|             Close|  Volume|ticker_code| ingestion_timestamp|
+--------------------+------------------+------------------+------------------+------------------+--------+-----------+--------------------+
|2025-08-11 00:00:...|28.229615147413917|28.523770873405553|  28.1101157534923|28.238807678222656|28045800|   PETR4.SA|2026-08-11 20:08:...|
|2025-08-12 00:00:...|28.303

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

PATH_BRONZE = "./storage/bronze"
PATH_SILVER = "./storage/silver"

print("🧹 Lendo Camada Bronze do Delta Lake...")
df_bronze = spark.read.format("delta").load(PATH_BRONZE)

# 1. Limpeza e Padronização de Tipos (Cleansing)
df_clean = df_bronze \
    .withColumn("date",        F.to_date(F.col("Date"))) \
    .withColumn("open_price",  F.round(F.col("Open").cast("double"), 4)) \
    .withColumn("high_price",  F.round(F.col("High").cast("double"), 4)) \
    .withColumn("low_price",   F.round(F.col("Low").cast("double"), 4)) \
    .withColumn("close_price", F.round(F.col("Close").cast("double"), 4)) \
    .withColumn("volume",      F.col("Volume").cast("long")) \
    .drop("Date", "Open", "High", "Low", "Close", "Volume") \
    .dropDuplicates(["ticker_code", "date"]) \
    .filter(F.col("close_price").isNotNull())

# 2. Especificação das Janelas Deslizantes por Ativo (Window Specs)
w21  = Window.partitionBy("ticker_code").orderBy("date").rowsBetween(-20, 0)
w200 = Window.partitionBy("ticker_code").orderBy("date").rowsBetween(-199, 0)

# 3. Engenharia de Recursos (Feature Engineering com Window Functions em PySpark)
df_silver = df_clean \
    .withColumn("daily_return_pct", F.round(((F.col("close_price") - F.col("open_price")) / F.col("open_price")) * 100, 4)) \
    .withColumn("sma_21",  F.round(F.avg("close_price").over(w21), 4)) \
    .withColumn("sma_200", F.round(F.avg("close_price").over(w200), 4)) \
    .withColumn("year",    F.year("date")) \
    .withColumn("month",   F.month("date"))

# 4. Salvar na Camada Silver como Delta Table (Particionado por Ticker)
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("ticker_code") \
    .save(PATH_SILVER)

print(f"✨ CAMADA SILVER CONCLUÍDA COM SUCESSO!")
print(f"📊 Total de registros enriquecidos: {df_silver.count()}")

# 5. Inspecionar o Schema enriquecido e os indicadores no PySpark
df_silver.printSchema()
df_silver.select("ticker_code", "date", "close_price", "daily_return_pct", "sma_21", "sma_200").show(5)


🧹 Lendo Camada Bronze do Delta Lake...


AnalysisException: Cannot resolve column name "date" among (ticker_code, ingestion_timestamp, open_price, high_price, low_price, close_price).